# 3.10 — Elastic Net

Elastic Net is linear regression with a deliberately mixed penalty: L1 pressure can set coefficients exactly to zero, while L2 pressure keeps correlated features from fighting too violently. In this notebook, we build that objective from scratch with NumPy, inspect the penalty pieces, and use validation-style decision scores to choose stable models rather than flattering training fits.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Elastic Net one idea at a time. Run each cell in order and read the printed intermediate values — the loss, L1 penalty, L2 penalty, mixed score, and validation decision are all exposed. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, least-squares pieces, and deterministic simulations.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic data and coordinate descent starts.

### 1. Empirical risk starts with residuals

Elastic Net still begins like ordinary linear regression: choose coefficients, predict with `X @ beta`, and measure residuals `y - X @ beta`. The empirical risk is the average evidence from the sample; the lesson's tiny verified losses are 0.191, 0.148, and 0.539, whose average is 0.293. That raw number is useful, but it is not yet the full model-selection score.

In [ ]:
losses_w = np.array([0.191, 0.148, 0.539])  # verified per-example losses from the lesson prose.
risk_w = float(losses_w.mean())  # empirical risk averages example losses.
print("losses:", losses_w)  # inspect each example's contribution.
print("empirical risk:", round(risk_w, 3))  # (0.191 + 0.148 + 0.539) / 3.
assert round(risk_w, 3) == 0.293  # matches the lesson arithmetic.

▶ What you'll see: three small losses averaged into the empirical training score 0.293.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["ex1", "ex2", "ex3"], losses_w, color="steelblue")
plt.axhline(risk_w, color="black", linestyle="--", label=f"mean={risk_w:.3f}")
plt.title("1: empirical risk is an average")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: the third example is the largest contributor, while the dashed line marks the averaged risk.

*Why it's done this way:* ERM optimizes an average because each example is one noisy glimpse of future data. A single low residual can be accidental; averaging forces the model to account for the whole sample before we add the regularization cost that guards generalization.

### 2. The L1 part selects features

The L1 norm is the sum of absolute coefficient sizes, `||beta||_1 = sum |beta_j|`. Its special behavior is the sharp corner at zero: when the data signal for a coordinate is weak, the optimum can land exactly on zero, removing that feature. That is why Elastic Net inherits lasso-style feature selection.

In [ ]:
beta_w = np.array([1.20, -0.50, 0.08, 0.00])  # four current linear coefficients.
l1_w = float(np.sum(np.abs(beta_w)))  # L1 norm adds absolute values.
print("beta:", beta_w)
print("|beta|:", np.abs(beta_w))
print("L1 norm:", round(l1_w, 3))
assert round(l1_w, 3) == 1.780

▶ What you'll see: positive and negative coefficients both add positive cost, while zero adds none.

In [ ]:
grid_w = np.linspace(-1.5, 1.5, 121)  # possible values for one coefficient.
l1_curve_w = np.abs(grid_w)  # L1 penalty for one coefficient.
plt.figure(figsize=(4.4, 3))
plt.plot(grid_w, l1_curve_w, color="darkorange")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("2: L1 has a corner at zero")
plt.xlabel("one coefficient")
plt.ylabel("absolute value")
plt.show()

▶ What you'll see: a V-shaped penalty whose kink is exactly at zero.

*Why it's done this way:* The V-shape means the penalty's slope changes abruptly at zero, so a small feature signal can be fully balanced by the penalty and stay at zero. That exact-zero possibility is the mathematical reason L1 behaves like a selector rather than merely a shrinker.

### 3. The L2 part shrinks and groups correlated features

The L2 term is one half of the squared norm, `0.5 * ||beta||_2^2`. It does not have a corner, so it usually shrinks coefficients continuously instead of setting them exactly to zero. Its strength is stability: spreading weight over correlated predictors is cheaper than making one coefficient huge and the other tiny.

In [ ]:
same_sum_w = np.array([[2.0, 0.0], [1.0, 1.0], [1.5, 0.5]])  # alternatives with similar combined signal.
l2_costs_w = 0.5 * np.sum(same_sum_w ** 2, axis=1)  # L2 cost for each split.
l1_costs_w = np.sum(np.abs(same_sum_w), axis=1)  # L1 cost is identical here.
print("coefficient splits:\n", same_sum_w)
print("L1 costs:", l1_costs_w)
print("0.5 L2 costs:", l2_costs_w)
assert np.allclose(l1_costs_w, [2.0, 2.0, 2.0])
assert np.allclose(l2_costs_w, [2.0, 1.0, 1.25])

▶ What you'll see: L1 is indifferent among these equal-sum splits, while L2 prefers the shared `[1, 1]` split.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["[2,0]", "[1,1]", "[1.5,.5]"], l2_costs_w, color="seagreen")
plt.title("3: L2 rewards sharing weight")
plt.ylabel("0.5 ||beta||²")
plt.show()

▶ What you'll see: the balanced split has the smallest L2 penalty even though all splits have the same total coefficient sum.

*Why it's done this way:* Squaring makes large single coefficients disproportionately expensive. For correlated columns that can explain the same signal, L2 discourages an unstable winner-take-all choice and nudges the model toward grouped, less erratic coefficients.

### 4. Elastic Net mixes the two penalties

Elastic Net uses the objective `0.5||y-Xβ||² + λ(α||β||₁ + (1-α)0.5||β||²)`. The parameter `alpha` chooses the blend: `alpha=1` is lasso, `alpha=0` is ridge, and middle values inherit selection pressure plus grouping stability.

In [ ]:
alpha_w = 0.6  # 60% L1 behavior, 40% L2 behavior.
lam_w = 0.10  # overall regularization strength.
l1_part_w = alpha_w * l1_w  # weighted L1 contribution inside the penalty.
l2_part_w = (1 - alpha_w) * 0.5 * float(np.sum(beta_w ** 2))  # weighted ridge contribution.
penalty_w = lam_w * (l1_part_w + l2_part_w)  # total Elastic Net cost.
print("weighted L1 part:", round(l1_part_w, 3))
print("weighted L2 part:", round(l2_part_w, 3))
print("Elastic Net penalty:", round(penalty_w, 3))
assert round(penalty_w, 3) == 0.141

▶ What you'll see: the mixed penalty is one number built from the L1 selector and L2 shrinker.

In [ ]:
alphas_w = np.linspace(0, 1, 6)  # ridge-to-lasso blend values.
penalties_w = lam_w * (alphas_w * l1_w + (1 - alphas_w) * 0.5 * np.sum(beta_w ** 2))
print("alpha grid:", alphas_w)
print("penalties:", np.round(penalties_w, 3))
plt.figure(figsize=(4.6, 3))
plt.plot(alphas_w, penalties_w, marker="o", color="purple")
plt.title("4: alpha changes the penalty blend")
plt.xlabel("alpha (0=ridge, 1=lasso)")
plt.ylabel("penalty")
plt.show()

▶ What you'll see: the cost changes smoothly as the blend moves from ridge-like to lasso-like.

*Why it's done this way:* `lambda` controls how much the model pays for complexity, while `alpha` controls what kind of complexity is discouraged. Elastic Net is useful because those two decisions are separated: overall restraint and selector-versus-grouper behavior are tuned independently.

### 5. Coordinate descent shows soft-thresholding plus shrinkage

A simple way to fit Elastic Net is coordinate descent: update one coefficient while holding the others fixed. For standardized columns, each update takes a correlation-like number `rho`, soft-thresholds it by the L1 amount, then divides by a denominator enlarged by the L2 amount.

In [ ]:
def soft_w(z_w, gamma_w):
    return np.sign(z_w) * max(abs(z_w) - gamma_w, 0.0)

rho_values_w = np.array([-0.20, -0.05, 0.04, 0.18, 0.60])  # one-coordinate data signals.
gamma_w = 0.10  # L1 threshold lambda*alpha.
soft_values_w = np.array([soft_w(r_w, gamma_w) for r_w in rho_values_w])
print("rho:", rho_values_w)
print("soft-thresholded:", np.round(soft_values_w, 3))
assert np.allclose(soft_values_w, [-0.10, 0.0, 0.0, 0.08, 0.50])

▶ What you'll see: weak signals inside ±0.10 become exactly zero.

In [ ]:
l2_denom_w = 1.0 + 0.08  # standardized column norm plus ridge shrinkage.
updated_w = soft_values_w / l2_denom_w  # Elastic Net coordinate update.
print("after L2 denominator:", np.round(updated_w, 3))
plt.figure(figsize=(4.8, 3))
plt.plot(rho_values_w, updated_w, marker="o", color="crimson")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvspan(-gamma_w, gamma_w, color="gray", alpha=0.2, label="zero zone")
plt.title("5: Elastic Net coordinate update")
plt.xlabel("rho signal")
plt.ylabel("new coefficient")
plt.legend()
plt.show()

▶ What you'll see: a flat zero region around the origin, then shrunken nonzero coefficients outside it.

*Why it's done this way:* The L1 threshold asks whether the coordinate's signal is strong enough to justify inclusion at all. The L2 denominator then shrinks any surviving coefficient, which reduces variance and stabilizes correlated predictors.

### 6. Model selection uses the full decision score

The lesson's arithmetic is a decision rule, not just a formula display. The raw empirical risk is 0.293, the method cost is 0.090, so the baseline decision score is 0.383. A tempting flexible alternative scores 0.419, while a stabilizing knob reduces the baseline by 20% to 0.306.

In [ ]:
cost_w = 0.090  # lesson complexity/regularization/operational cost.
score_w = round(risk_w, 3) + cost_w  # full score, using the rounded lesson risk before adding cost.
alt_w = 0.419  # tempting flexible alternative.
gap_w = alt_w - score_w
rel_gap_w = gap_w / alt_w
stable_w = 0.80 * score_w
print("baseline score:", round(score_w, 3))
print("gap:", round(gap_w, 3), "relative gap:", round(rel_gap_w, 3))
print("stabilized score:", round(stable_w, 3))
assert round(score_w, 3) == 0.383
assert round(gap_w, 3) == 0.036
assert round(rel_gap_w, 3) == 0.086
assert round(stable_w, 3) == 0.306

▶ What you'll see: the full decision score includes cost, and the stabilized setting is lower than both alternatives.

In [ ]:
names_w = ["raw risk", "risk+cost", "flexible alt", "stabilized"]
values_w = [risk_w, score_w, alt_w, stable_w]
plt.figure(figsize=(5.2, 3))
plt.bar(names_w, values_w, color=["gray", "steelblue", "darkorange", "seagreen"])
plt.title("6: select by the full score")
plt.ylabel("decision score (lower is better)")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: the raw risk looks best only because it ignores cost; among valid decision scores, stabilized wins.

*Why it's done this way:* Regularization is a generalization bargain. We do not choose the prettiest training fragment; we choose the option whose complete score includes fit, complexity, and stability, because that is the quantity most aligned with future performance.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each Elastic Net mechanic by hand.** These new tiny NumPy-only toys
> isolate the residual, L1, L2, mixed-penalty, coordinate-update, and full-score computations.
> Run them top to bottom: every block prints intermediates, draws one picture, and includes an `assert`.

### ✍️ Toy 1 · Residuals create empirical risk

Elastic Net starts with ordinary regression residuals. Square each residual, then average those losses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_y = np.array([1.0, 2.0, 2.0, 3.0, 5.0, 6.0])
print("targets:", t1_y.tolist())  # -> [1.0, 2.0, 2.0, 3.0, 5.0, 6.0]
t1_pred = np.array([1.1, 1.8, 2.4, 2.6, 4.7, 6.5])
print("predictions:", t1_pred.tolist())  # -> [1.1, 1.8, 2.4, 2.6, 4.7, 6.5]
t1_resid = t1_y - t1_pred  # -> [-0.1  0.2 -0.4  0.4  0.3 -0.5]
print("residuals:", np.round(t1_resid, 3).tolist())  # -> [-0.1, 0.2, -0.4, 0.4, 0.3, -0.5]
t1_losses = t1_resid ** 2  # -> [0.01 0.04 0.16 0.16 0.09 0.25]
print("squared losses:", np.round(t1_losses, 3).tolist())  # -> [0.01, 0.04, 0.16, 0.16, 0.09, 0.25]
t1_risk = float(t1_losses.mean())  # -> 0.11833333333333333
print("empirical risk:", round(t1_risk, 4))  # -> 0.1183
assert abs(t1_risk - 0.11833333333333333) < 1e-12

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t1_losses.size), t1_losses, color="steelblue")
plt.axhline(t1_risk, color="crimson", linestyle="--", label=f"mean={t1_risk:.4f}")
plt.xlabel("example")
plt.ylabel("squared loss")
plt.title("Toy 1 · residuals become risk")
plt.legend()
plt.show()

▶ What you'll see: the squared-loss bars average to `0.1183`.

### ✍️ Toy 2 · L1 counts absolute coefficient mass

The L1 part charges positive and negative coefficients equally, while exact zeros add no cost and mark inactive features.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_beta = np.array([1.2, -0.7, 0.0, 0.1, 0.0, -0.3])
print("beta:", t2_beta.tolist())  # -> [1.2, -0.7, 0.0, 0.1, 0.0, -0.3]
t2_abs = np.abs(t2_beta)  # -> [1.2 0.7 0.  0.1 0.  0.3]
print("absolute values:", t2_abs.tolist())  # -> [1.2, 0.7, 0.0, 0.1, 0.0, 0.3]
t2_l1 = float(t2_abs.sum())  # -> 2.3
print("L1 norm:", round(t2_l1, 3))  # -> 2.3
t2_active = t2_abs > 0.0  # -> [ True  True False  True False  True]
print("active mask:", t2_active.astype(int).tolist())  # -> [1, 1, 0, 1, 0, 1]
t2_active_count = int(t2_active.sum())  # -> 4
print("active count:", t2_active_count)  # -> 4
assert t2_l1 == 2.3 and t2_active_count == 4

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t2_beta.size), t2_beta, color=["seagreen" if a else "lightgray" for a in t2_active])
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("coefficient index")
plt.ylabel("coefficient")
plt.title("Toy 2 · zeros are inactive")
plt.show()

▶ What you'll see: two gray bars sit at zero, while four nonzero coefficients pay L1 cost.

### ✍️ Toy 3 · L2 prefers shared weight across correlated features

For equal-sum coefficient splits, L1 is indifferent, but the L2 half-squared norm is smallest when weight is shared.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_splits = np.array([[3.0, 0.0], [1.5, 1.5], [2.0, 1.0]])
print("coefficient splits:", t3_splits.tolist())  # -> [[3.0, 0.0], [1.5, 1.5], [2.0, 1.0]]
t3_l1 = np.sum(np.abs(t3_splits), axis=1)  # -> [3. 3. 3.]
print("L1 costs:", t3_l1.tolist())  # -> [3.0, 3.0, 3.0]
t3_half_l2 = 0.5 * np.sum(t3_splits ** 2, axis=1)  # -> [4.5  2.25 2.5 ]
print("0.5 L2 costs:", t3_half_l2.tolist())  # -> [4.5, 2.25, 2.5]
t3_best = int(np.argmin(t3_half_l2))  # -> 1
print("lowest-L2 split:", t3_splits[t3_best].tolist())  # -> [1.5, 1.5]
assert np.allclose(t3_l1, [3.0, 3.0, 3.0]) and t3_best == 1

plt.figure(figsize=(4.8, 2.8))
plt.bar(["[3,0]", "[1.5,1.5]", "[2,1]"], t3_half_l2, color=["gray", "seagreen", "gray"])
plt.ylabel("0.5 ||β||²")
plt.title("Toy 3 · L2 rewards sharing")
plt.show()

▶ What you'll see: the balanced split has the lowest L2 cost even though all splits have the same L1 cost.

### ✍️ Toy 4 · Alpha blends L1 and L2 into one penalty

Elastic Net computes the L1 part, the L2 part, blends them with `alpha`, then scales by `lambda`.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_beta = np.array([0.8, -0.4, 0.2, 0.0, 0.5, -0.1])
print("beta:", t4_beta.tolist())  # -> [0.8, -0.4, 0.2, 0.0, 0.5, -0.1]
t4_l1 = float(np.sum(np.abs(t4_beta)))  # -> 2.0
print("L1 norm:", round(t4_l1, 3))  # -> 2.0
t4_half_l2 = 0.5 * float(np.sum(t4_beta ** 2))  # -> 0.55
print("0.5 L2 norm:", round(t4_half_l2, 3))  # -> 0.55
t4_alpha = 0.7
print("alpha:", t4_alpha)  # -> 0.7
t4_lam = 0.2
print("lambda:", t4_lam)  # -> 0.2
t4_weighted_l1 = t4_alpha * t4_l1  # -> 1.4
print("weighted L1:", round(t4_weighted_l1, 3))  # -> 1.4
t4_weighted_l2 = (1.0 - t4_alpha) * t4_half_l2  # -> 0.165
print("weighted L2:", round(t4_weighted_l2, 3))  # -> 0.165
t4_inside = t4_weighted_l1 + t4_weighted_l2  # -> 1.565
print("inside blend:", round(t4_inside, 3))  # -> 1.565
t4_penalty = t4_lam * t4_inside  # -> 0.313
print("Elastic Net penalty:", round(t4_penalty, 3))  # -> 0.313
assert round(t4_penalty, 3) == 0.313

plt.figure(figsize=(4.8, 2.8))
plt.bar(["α||β||₁", "(1-α)0.5||β||²", "λ blend"], [t4_weighted_l1, t4_weighted_l2, t4_penalty], color=["orange", "seagreen", "purple"])
plt.xticks(rotation=12)
plt.ylabel("penalty piece")
plt.title("Toy 4 · L1/L2 blend")
plt.show()

▶ What you'll see: the blended inside penalty is mostly L1 here, then `lambda` scales it down to `0.313`.

### ✍️ Toy 5 · Elastic Net coordinate update thresholds then shrinks

A coordinate signal first passes through the L1 zero zone. Any survivor is then divided by the L2-enlarged denominator.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_rho = np.array([-0.6, -0.15, -0.04, 0.08, 0.32, 0.9])
print("rho signals:", t5_rho.tolist())  # -> [-0.6, -0.15, -0.04, 0.08, 0.32, 0.9]
t5_gamma = 0.2
print("L1 threshold:", t5_gamma)  # -> 0.2
t5_soft_raw = np.sign(t5_rho) * np.maximum(np.abs(t5_rho) - t5_gamma, 0.0)
t5_soft = np.where(np.isclose(t5_soft_raw, 0.0), 0.0, t5_soft_raw)  # -> [-0.4   0.    0.    0.    0.12  0.7 ]
print("soft-thresholded:", np.round(t5_soft, 3).tolist())  # -> [-0.4, 0.0, 0.0, 0.0, 0.12, 0.7]
t5_denom = 1.3
print("L2 denominator:", round(t5_denom, 3))  # -> 1.3
t5_update = t5_soft / t5_denom  # -> [-0.30769231  0.          0.          0.          0.09230769  0.53846154]
print("coordinate updates:", np.round(t5_update, 3).tolist())  # -> [-0.308, 0.0, 0.0, 0.0, 0.092, 0.538]
assert np.allclose(np.round(t5_update, 3), [-0.308, 0.0, 0.0, 0.0, 0.092, 0.538])

plt.figure(figsize=(5.0, 2.8))
plt.plot(t5_rho, t5_update, marker="o", color="crimson")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvspan(-t5_gamma, t5_gamma, color="gray", alpha=0.2, label="zero zone")
plt.xlabel("rho signal")
plt.ylabel("updated coefficient")
plt.title("Toy 5 · threshold then shrink")
plt.legend()
plt.show()

▶ What you'll see: small signals map to zero; nonzero updates are smaller than their soft-thresholded values.

### ✍️ Toy 6 · Select by the full Elastic Net decision score

Risk, penalty, and stability credit all live in the final comparison. The mixed setting wins only after all pieces are included.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_labels = np.array(["ridge-like", "lasso-like", "mixed"])
print("labels:", t6_labels.tolist())  # -> ['ridge-like', 'lasso-like', 'mixed']
t6_risk = np.array([0.20, 0.17, 0.18])
print("risks:", np.round(t6_risk, 3).tolist())  # -> [0.2, 0.17, 0.18]
t6_penalty = np.array([0.07, 0.14, 0.09])
print("penalties:", np.round(t6_penalty, 3).tolist())  # -> [0.07, 0.14, 0.09]
t6_credit = np.array([0.00, 0.02, 0.05])
print("stability credits:", np.round(t6_credit, 3).tolist())  # -> [0.0, 0.02, 0.05]
t6_score = t6_risk + t6_penalty - t6_credit  # -> [0.27 0.29 0.22]
print("full scores:", np.round(t6_score, 3).tolist())  # -> [0.27, 0.29, 0.22]
t6_best = int(np.argmin(t6_score))  # -> 2
print("winner:", t6_labels[t6_best])  # -> mixed
assert t6_labels[t6_best] == "mixed"

plt.figure(figsize=(4.8, 2.8))
plt.bar(t6_labels, t6_score, color=["gray", "orange", "seagreen"])
plt.ylabel("decision score")
plt.title("Toy 6 · full score chooses mixed")
plt.show()

▶ What you'll see: the mixed bar is lowest after risk, penalty, and stability are combined.

## 🛠️ Setup

In [ ]:
import numpy as np  # Import NumPy for arrays, least squares, coordinate descent, and numerical checks.
import matplotlib.pyplot as plt  # Import Matplotlib for coefficient paths, loss curves, and validation plots.
np.random.seed(0)  # Fix the global seed so stochastic examples are reproducible.

def en_penalty(beta, lam, alpha):  # Compute the Elastic Net penalty λ(α||β||₁ + (1-α)0.5||β||²).
    beta = np.asarray(beta, dtype=float)  # Convert coefficients to a predictable float array.
    return float(lam * (alpha * np.sum(np.abs(beta)) + (1 - alpha) * 0.5 * np.sum(beta ** 2)))  # Return the mixed penalty.

def en_objective(X, y, beta, lam, alpha):  # Compute squared-error objective plus Elastic Net penalty.
    residual = y - X @ beta  # Compute prediction errors for the current coefficients.
    return float(0.5 * np.sum(residual ** 2) + en_penalty(beta, lam, alpha))  # Add data fit and complexity cost.

def soft_threshold(z, gamma):  # Apply the lasso soft-threshold operator.
    return np.sign(z) * np.maximum(np.abs(z) - gamma, 0.0)  # Shrink toward zero and set weak signals exactly to zero.

def standardize(X):  # Center and scale columns for comparable penalties.
    mean = X.mean(axis=0)  # Column means.
    scale = X.std(axis=0)  # Column standard deviations.
    scale = np.where(scale == 0, 1.0, scale)  # Guard against constant columns.
    return (X - mean) / scale, mean, scale  # Return standardized design and transformation pieces.

def fit_elastic_net_cd(X, y, lam=0.1, alpha=0.5, steps=400):  # Small coordinate-descent solver for standardized data.
    beta = np.zeros(X.shape[1])  # Start all coefficients at zero.
    losses = []  # Store objective values for inspection.
    for step in range(steps):  # Repeat coordinate updates.
        for j in range(X.shape[1]):  # Update one coefficient at a time.
            residual = y - X @ beta + X[:, j] * beta[j]  # Add back feature j's current contribution.
            rho = float(X[:, j] @ residual)  # Correlation of feature j with the partial residual.
            denom = float(np.sum(X[:, j] ** 2) + lam * (1 - alpha))  # Squared column size plus L2 shrinkage.
            beta[j] = soft_threshold(rho, lam * alpha) / denom  # L1 threshold then L2 shrink.
        if step % 20 == 0 or step == steps - 1:  # Record a compact convergence trace.
            losses.append(en_objective(X, y, beta, lam, alpha))  # Save the full objective.
    return beta, np.array(losses)  # Return fitted coefficients and objective curve.

## 🟢 Basics (warm-up)

### Basic 1 — Average the empirical losses

**Goal.** Recompute the lesson's raw empirical risk, because Elastic Net still begins with ordinary training loss before adding a penalty. We build it in 2 steps.

In [ ]:
losses_b1 = np.array([0.191, 0.148, 0.539])  # Store the verified per-example losses from lesson 3.10.
print("losses_b1:", losses_b1)  # Inspect the individual training losses.

▶ What you'll see: three per-example losses that will be averaged into the empirical risk.

In [ ]:
risk_b1 = float(np.mean(losses_b1))  # Average losses to get empirical risk R_S.
print("R_S:", round(risk_b1, 3))  # Inspect the raw training score.
assert round(risk_b1, 3) == 0.293  # Verify the lesson arithmetic.
plt.figure(figsize=(4, 3))
plt.bar(["1", "2", "3"], losses_b1, color="steelblue")
plt.axhline(risk_b1, color="black", linestyle="--")
plt.title("Basic 1: empirical risk")
plt.ylabel("loss")
plt.show()

▶ What you'll see: the dashed mean summarizes all three losses.

👀 Takeaway: empirical risk is the average fit term that regularization will augment.

### Basic 2 — Add the method cost

**Goal.** Add the lesson's 0.090 cost to the raw risk, because model selection should use the full score rather than training loss alone. We build it in 2 steps.

In [ ]:
risk_b2 = 0.293  # Use the rounded lesson empirical risk.
cost_b2 = 0.090  # Use the verified complexity or regularization cost.
print("risk:", risk_b2, "cost:", cost_b2)  # Inspect both pieces before adding.

▶ What you'll see: the score has a fit piece and a cost piece.

In [ ]:
score_b2 = risk_b2 + cost_b2  # Compute the full selection score.
print("risk + cost:", round(score_b2, 3))  # Inspect the decision score.
assert round(score_b2, 3) == 0.383  # Verify the lesson number.
plt.figure(figsize=(4, 3))
plt.bar(["risk", "cost", "total"], [risk_b2, cost_b2, score_b2], color=["gray", "orange", "seagreen"])
plt.title("Basic 2: full score")
plt.ylabel("score")
plt.show()

▶ What you'll see: the total bar is taller than raw risk because it includes complexity cost.

👀 Takeaway: dropping the cost changes the decision rule Elastic Net is meant to enforce.

### Basic 3 — Compute an L1 norm

**Goal.** Measure coefficient size with absolute values, because the L1 piece is what can push weak coefficients exactly to zero. We build it in 2 steps.

In [ ]:
beta_b3 = np.array([1.2, -0.5, 0.0, 0.3])  # Define a small coefficient vector with positive, negative, and zero entries.
abs_beta_b3 = np.abs(beta_b3)  # Convert all coefficient magnitudes to nonnegative costs.
print("abs coefficients:", abs_beta_b3)  # Inspect each L1 contribution.

▶ What you'll see: signs disappear because L1 charges magnitude, not direction.

In [ ]:
l1_b3 = float(np.sum(abs_beta_b3))  # Sum absolute values to get ||beta||_1.
print("L1 norm:", round(l1_b3, 3))  # Inspect the selector-style penalty size.
assert round(l1_b3, 3) == 2.000  # Verify the simple hand calculation.
plt.figure(figsize=(4, 3))
plt.bar(["b0", "b1", "b2", "b3"], abs_beta_b3, color="darkorange")
plt.title("Basic 3: L1 contributions")
plt.ylabel("|coefficient|")
plt.show()

▶ What you'll see: each bar contributes linearly to the L1 norm.

👀 Takeaway: L1 penalizes total absolute coefficient mass and can reward exact sparsity.

### Basic 4 — Compute an L2 penalty

**Goal.** Square coefficient sizes, because the ridge part makes large coefficients increasingly expensive. We build it in 2 steps.

In [ ]:
beta_b4 = np.array([1.2, -0.5, 0.0, 0.3])  # Reuse a small coefficient vector.
squares_b4 = beta_b4 ** 2  # Square each coefficient for the L2 penalty.
print("squared coefficients:", np.round(squares_b4, 3))  # Inspect how large coefficients dominate.

▶ What you'll see: the largest coefficient contributes much more after squaring.

In [ ]:
half_l2_b4 = 0.5 * float(np.sum(squares_b4))  # Compute 0.5||beta||_2^2 as used in the Elastic Net formula.
print("0.5 L2 squared norm:", round(half_l2_b4, 3))  # Inspect the ridge-style penalty size.
assert round(half_l2_b4, 3) == 0.890  # Verify 0.5*(1.44+0.25+0+0.09).
plt.figure(figsize=(4, 3))
plt.bar(["b0²", "b1²", "b2²", "b3²"], squares_b4, color="seagreen")
plt.title("Basic 4: L2 squared pieces")
plt.ylabel("coefficient²")
plt.show()

▶ What you'll see: the squared penalty concentrates cost on large coordinates.

👀 Takeaway: L2 shrinks coefficients smoothly by making large magnitudes costly.

### Basic 5 — Blend L1 and L2

**Goal.** Compute the Elastic Net penalty from `lambda` and `alpha`, because those two knobs separate total strength from penalty type. We build it in 2 steps.

In [ ]:
beta_b5 = np.array([1.2, -0.5, 0.0, 0.3])  # Define coefficients to penalize.
lam_b5 = 0.1  # Set overall penalty strength.
alpha_b5 = 0.6  # Choose a blend closer to lasso than ridge.
print("lambda:", lam_b5, "alpha:", alpha_b5)  # Inspect the hyperparameters.

▶ What you'll see: lambda controls size, alpha controls the mixture.

In [ ]:
pen_b5 = en_penalty(beta_b5, lam_b5, alpha_b5)  # Compute λ(α||β||₁ + (1-α)0.5||β||²).
print("Elastic Net penalty:", round(pen_b5, 3))  # Inspect the mixed regularization cost.
assert round(pen_b5, 3) == 0.156  # Verify the blended penalty calculation.
plt.figure(figsize=(4, 3))
plt.bar(["L1 share", "L2 share"], [alpha_b5 * np.sum(np.abs(beta_b5)), (1 - alpha_b5) * 0.5 * np.sum(beta_b5 ** 2)], color=["orange", "green"])
plt.title("Basic 5: penalty blend before λ")
plt.ylabel("unscaled contribution")
plt.show()

▶ What you'll see: the L1 share is larger here because alpha is 0.6.

👀 Takeaway: Elastic Net is a weighted contract between selection and shrinkage.

### Basic 6 — Draw the soft-threshold rule

**Goal.** Apply the L1 soft-threshold operator, because it is the tiny arithmetic move that creates exact zeros. We build it in 2 steps.

In [ ]:
z_b6 = np.array([-0.3, -0.08, 0.0, 0.07, 0.4])  # Define possible one-coordinate signals.
gamma_b6 = 0.1  # Define the L1 threshold.
soft_b6 = soft_threshold(z_b6, gamma_b6)  # Shrink signals toward zero and zero out weak ones.
print("signals:", z_b6)  # Inspect raw signals.
print("soft-thresholded:", soft_b6)  # Inspect thresholded coefficients.

▶ What you'll see: signals whose magnitude is below 0.1 become zero.

In [ ]:
assert np.allclose(soft_b6, [-0.2, 0.0, 0.0, 0.0, 0.3])  # Verify the thresholding result.
plt.figure(figsize=(4, 3))
plt.plot(z_b6, soft_b6, marker="o", color="crimson")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 6: soft threshold")
plt.xlabel("signal")
plt.ylabel("coefficient after L1")
plt.show()

▶ What you'll see: a flat zero region near the origin.

👀 Takeaway: L1 does feature selection by deleting weak coordinate signals.

### Basic 7 — Standardize feature scales

**Goal.** Center and scale features, because regularization should compare coefficients on a fair feature scale. We build it in 2 steps.

In [ ]:
X_b7 = np.array([[1.0, 100.0], [2.0, 120.0], [3.0, 140.0], [4.0, 160.0]])  # Two features with very different units.
Xs_b7, mean_b7, scale_b7 = standardize(X_b7)  # Put columns on comparable scales.
print("means:", mean_b7)  # Inspect the centering constants.
print("scales:", np.round(scale_b7, 3))  # Inspect the scaling constants.

▶ What you'll see: the second feature's raw unit scale is much larger.

In [ ]:
print("standardized means:", np.round(Xs_b7.mean(axis=0), 6))  # Check centering.
print("standardized stds:", np.round(Xs_b7.std(axis=0), 6))  # Check scaling.
assert np.allclose(Xs_b7.mean(axis=0), [0.0, 0.0])
assert np.allclose(Xs_b7.std(axis=0), [1.0, 1.0])
plt.figure(figsize=(4, 3))
plt.scatter(Xs_b7[:, 0], Xs_b7[:, 1], color="purple")
plt.title("Basic 7: standardized features")
plt.xlabel("feature 0 scaled")
plt.ylabel("feature 1 scaled")
plt.show()

▶ What you'll see: both columns now live on the same standardized scale.

👀 Takeaway: without standardization, a penalty can punish units rather than importance.

### Basic 8 — Compute predictions and residuals

**Goal.** Turn coefficients into predictions and errors, because the data-fit term is built from residuals. We build it in 2 steps.

In [ ]:
X_b8 = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])  # Include an intercept-like column and one feature.
y_b8 = np.array([1.0, 2.0, 2.5])  # Define three observed targets.
beta_b8 = np.array([1.0, 0.7])  # Choose a candidate linear rule.
pred_b8 = X_b8 @ beta_b8  # Predict y from X beta.
print("predictions:", pred_b8)  # Inspect fitted values.

▶ What you'll see: the linear rule makes one prediction per row.

In [ ]:
resid_b8 = y_b8 - pred_b8  # Compute residuals.
sse_b8 = 0.5 * float(np.sum(resid_b8 ** 2))  # Compute half squared error.
print("residuals:", np.round(resid_b8, 3))  # Inspect signed errors.
print("0.5 SSE:", round(sse_b8, 3))  # Inspect the fit term.
assert round(sse_b8, 3) == 0.050
plt.figure(figsize=(4, 3))
plt.bar(["row0", "row1", "row2"], resid_b8, color="steelblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 8: residuals")
plt.ylabel("y - prediction")
plt.show()

▶ What you'll see: residual bars show which rows are under- or over-predicted.

👀 Takeaway: Elastic Net changes linear regression by adding a penalty to this residual fit term.

### Basic 9 — Compare ridge, lasso, and Elastic Net settings

**Goal.** Evaluate the same coefficients under three alpha values, because alpha changes the kind of regularization while lambda stays fixed. We build it in 2 steps.

In [ ]:
beta_b9 = np.array([1.0, 0.5, 0.0])  # Define a small coefficient vector.
lam_b9 = 0.2  # Keep the overall penalty strength fixed.
alphas_b9 = np.array([0.0, 0.5, 1.0])  # Ridge, Elastic Net middle, lasso.
labels_b9 = ["ridge α=0", "elastic α=.5", "lasso α=1"]  # Names for interpretation.
print("alphas:", alphas_b9)  # Inspect the compared settings.

▶ What you'll see: the grid spans pure L2 to pure L1.

In [ ]:
penalties_b9 = np.array([en_penalty(beta_b9, lam_b9, a_b9) for a_b9 in alphas_b9])  # Compute each penalty.
print("penalties:", np.round(penalties_b9, 3))  # Inspect the effect of alpha.
assert np.allclose(np.round(penalties_b9, 3), [0.125, 0.213, 0.300])
plt.figure(figsize=(5, 3))
plt.bar(labels_b9, penalties_b9, color=["green", "purple", "orange"])
plt.title("Basic 9: same beta, different alpha")
plt.ylabel("penalty")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: the penalty changes as the blend moves toward L1 for this coefficient vector.

👀 Takeaway: alpha is not cosmetic; it changes how coefficient complexity is priced.

### Basic 10 — Select the lowest full score

**Goal.** Compare baseline, flexible, and stabilized scores, because the lesson's final choice is made by the minimum full decision score. We build it in 2 steps.

In [ ]:
scores_b10 = np.array([0.383, 0.419, 0.306])  # Baseline, flexible alternative, and stabilized score.
names_b10 = np.array(["baseline", "flexible", "stabilized"])  # Label the options.
best_idx_b10 = int(np.argmin(scores_b10))  # Find the lowest score.
print("scores:", dict(zip(names_b10, scores_b10)))  # Inspect all options.

▶ What you'll see: the stabilized score is numerically smallest.

In [ ]:
print("best option:", names_b10[best_idx_b10], "score:", scores_b10[best_idx_b10])  # Report the selected model.
assert names_b10[best_idx_b10] == "stabilized"
assert round(float(scores_b10[best_idx_b10]), 3) == 0.306
plt.figure(figsize=(4.6, 3))
plt.bar(names_b10, scores_b10, color=["steelblue", "darkorange", "seagreen"])
plt.title("Basic 10: choose the minimum score")
plt.ylabel("lower is better")
plt.show()

▶ What you'll see: the green stabilized bar wins the full-score comparison.

👀 Takeaway: model selection should compare complete scores, not isolated training fragments.

## 🟡 Easy

### Easy 1 — Fit a tiny Elastic Net by coordinate descent

**Goal.** Train coefficients from scratch on standardized data, because the objective becomes concrete when we can watch its value decrease. We build it in 3 steps.

In [ ]:
x1_e1 = np.linspace(-2, 2, 12)  # Create one informative feature.
x2_e1 = x1_e1 + 0.15 * np.sin(np.arange(12))  # Create a correlated companion feature.
X_e1_raw = np.column_stack([x1_e1, x2_e1])  # Build a two-feature design matrix.
y_e1 = 1.5 * x1_e1 + 1.4 * x2_e1 + 0.2 * np.cos(np.arange(12))  # Create targets with both features useful.
X_e1, _, _ = standardize(X_e1_raw)  # Standardize before penalized fitting.
print("X shape:", X_e1.shape, "y shape:", y_e1.shape)  # Inspect problem size.

▶ What you'll see: a tiny two-feature regression problem with correlated columns.

In [ ]:
beta_e1, losses_e1 = fit_elastic_net_cd(X_e1, y_e1, lam=0.4, alpha=0.5, steps=300)  # Fit Elastic Net by coordinate descent.
print("beta:", np.round(beta_e1, 3))  # Inspect learned coefficients.
print("loss start -> end:", round(losses_e1[0], 3), "->", round(losses_e1[-1], 3))  # Inspect convergence.
assert losses_e1[-1] <= losses_e1[0]

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(np.arange(len(losses_e1)) * 20, losses_e1, color="purple")
plt.title("Easy 1: Elastic Net objective decreases")
plt.xlabel("coordinate-descent step")
plt.ylabel("objective")
plt.show()

▶ What you'll see: the objective drops quickly and then flattens as coordinate descent converges.

👀 Takeaway: Elastic Net fitting is ordinary residual reduction plus repeated penalty-aware coefficient updates.

### Easy 2 — Show lasso sparsity versus ridge shrinkage

**Goal.** Fit the same noisy problem with alpha 1 and alpha 0, because lasso tends to select while ridge tends to keep coefficients. We build it in 3 steps.

In [ ]:
rng_e2 = np.random.default_rng(2)  # Create deterministic noise.
X_e2_raw = rng_e2.normal(size=(40, 5))  # Generate five candidate features.
X_e2_raw[:, 1] = X_e2_raw[:, 0] + 0.05 * rng_e2.normal(size=40)  # Make feature 1 highly correlated with feature 0.
y_e2 = 2.0 * X_e2_raw[:, 0] - 1.0 * X_e2_raw[:, 3] + 0.2 * rng_e2.normal(size=40)  # Only features 0 and 3 truly matter.
X_e2, _, _ = standardize(X_e2_raw)  # Standardize before comparing penalties.
print("correlation x0,x1:", round(float(np.corrcoef(X_e2[:, 0], X_e2[:, 1])[0, 1]), 3))  # Inspect correlated predictors.

▶ What you'll see: features 0 and 1 are almost duplicates.

In [ ]:
beta_lasso_e2, _ = fit_elastic_net_cd(X_e2, y_e2, lam=1.0, alpha=1.0, steps=500)  # Pure L1 fit.
beta_ridge_e2, _ = fit_elastic_net_cd(X_e2, y_e2, lam=1.0, alpha=0.0, steps=500)  # Pure L2 fit.
print("lasso beta:", np.round(beta_lasso_e2, 3))  # Inspect sparse-ish coefficients.
print("ridge beta:", np.round(beta_ridge_e2, 3))  # Inspect smoother shrinkage.
assert np.count_nonzero(np.abs(beta_lasso_e2) < 1e-6) >= 1

In [ ]:
x_e2 = np.arange(5)
plt.figure(figsize=(5, 3))
plt.bar(x_e2 - 0.18, beta_lasso_e2, width=0.36, label="lasso α=1", color="orange")
plt.bar(x_e2 + 0.18, beta_ridge_e2, width=0.36, label="ridge α=0", color="green")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Easy 2: selection vs shrinkage")
plt.xlabel("feature")
plt.legend()
plt.show()

▶ What you'll see: lasso is more willing to zero out coefficients, while ridge spreads and shrinks weight.

👀 Takeaway: Elastic Net sits between these behaviors rather than choosing only one extreme.

### Easy 3 — Stabilize correlated features with a mixed penalty

**Goal.** Compare lasso and Elastic Net on correlated predictors, because Elastic Net's L2 part often keeps related features from behaving erratically. We build it in 3 steps.

In [ ]:
rng_e3 = np.random.default_rng(3)  # Create reproducible synthetic data.
z_e3 = rng_e3.normal(size=50)  # Shared latent signal.
X_e3_raw = np.column_stack([z_e3 + 0.03 * rng_e3.normal(size=50), z_e3 + 0.03 * rng_e3.normal(size=50), rng_e3.normal(size=50)])  # Two correlated useful features plus noise.
y_e3 = 1.2 * X_e3_raw[:, 0] + 1.2 * X_e3_raw[:, 1] + 0.1 * rng_e3.normal(size=50)  # Target depends on both correlated features.
X_e3, _, _ = standardize(X_e3_raw)  # Standardize for fair penalty treatment.
print("corr useful pair:", round(float(np.corrcoef(X_e3[:, 0], X_e3[:, 1])[0, 1]), 3))  # Inspect correlation.

▶ What you'll see: the first two features nearly duplicate the same signal.

In [ ]:
beta_lasso_e3, _ = fit_elastic_net_cd(X_e3, y_e3, lam=0.9, alpha=1.0, steps=500)  # Lasso fit.
beta_en_e3, _ = fit_elastic_net_cd(X_e3, y_e3, lam=0.9, alpha=0.4, steps=500)  # Mixed Elastic Net fit.
share_lasso_e3 = abs(beta_lasso_e3[0] - beta_lasso_e3[1])  # Measure imbalance between correlated features.
share_en_e3 = abs(beta_en_e3[0] - beta_en_e3[1])  # Measure Elastic Net imbalance.
print("lasso beta:", np.round(beta_lasso_e3, 3))
print("elastic beta:", np.round(beta_en_e3, 3))
print("imbalance lasso vs EN:", round(share_lasso_e3, 3), round(share_en_e3, 3))

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["lasso imbalance", "EN imbalance"], [share_lasso_e3, share_en_e3], color=["orange", "purple"])
plt.title("Easy 3: correlated-feature stability")
plt.ylabel("|β0 - β1|")
plt.show()

▶ What you'll see: the mixed penalty usually balances the correlated pair more than pure lasso.

👀 Takeaway: the L2 part of Elastic Net encourages grouping when predictors carry similar information.

### Easy 4 — Trace coefficients as lambda grows

**Goal.** Sweep lambda values, because stronger regularization should shrink coefficients and can eventually remove weak ones. We build it in 3 steps.

In [ ]:
rng_e4 = np.random.default_rng(4)  # Create deterministic data.
X_e4_raw = rng_e4.normal(size=(45, 4))  # Four features.
y_e4 = 1.8 * X_e4_raw[:, 0] - 0.6 * X_e4_raw[:, 2] + 0.2 * rng_e4.normal(size=45)  # Two useful signals.
X_e4, _, _ = standardize(X_e4_raw)  # Standardize before the lambda sweep.
lams_e4 = np.array([0.01, 0.1, 0.4, 1.0, 2.0])  # Increasing penalty strengths.
print("lambda grid:", lams_e4)  # Inspect sweep settings.

▶ What you'll see: the experiment moves from almost unregularized to strongly regularized.

In [ ]:
coefs_e4 = []  # Store one coefficient vector per lambda.
for lam_e4 in lams_e4:
    beta_e4, _ = fit_elastic_net_cd(X_e4, y_e4, lam=lam_e4, alpha=0.7, steps=500)  # Fit one Elastic Net model.
    coefs_e4.append(beta_e4)  # Save the fitted coefficients.
coefs_e4 = np.array(coefs_e4)  # Convert to an array for plotting.
print("coefs by lambda:\n", np.round(coefs_e4, 3))  # Inspect shrinkage numerically.
assert np.linalg.norm(coefs_e4[-1], 1) <= np.linalg.norm(coefs_e4[0], 1)

In [ ]:
plt.figure(figsize=(5, 3))
for j_e4 in range(coefs_e4.shape[1]):
    plt.plot(lams_e4, coefs_e4[:, j_e4], marker="o", label=f"β{j_e4}")
plt.xscale("log")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Easy 4: coefficient path")
plt.xlabel("lambda")
plt.ylabel("coefficient")
plt.legend()
plt.show()

▶ What you'll see: coefficients shrink toward zero as lambda increases.

👀 Takeaway: lambda is the main strength knob that controls coefficient complexity.

### Easy 5 — Validate alpha choices

**Goal.** Pick alpha by validation error, because the best lasso-ridge blend should be judged on data not used for fitting. We build it in 4 steps.

In [ ]:
rng_e5 = np.random.default_rng(5)  # Create reproducible data.
X_all_e5 = rng_e5.normal(size=(60, 5))  # Generate candidate features.
X_all_e5[:, 1] = X_all_e5[:, 0] + 0.1 * rng_e5.normal(size=60)  # Add a correlated feature.
y_all_e5 = 1.5 * X_all_e5[:, 0] + 1.0 * X_all_e5[:, 1] - 0.7 * X_all_e5[:, 4] + 0.3 * rng_e5.normal(size=60)  # Define targets.
X_all_e5, _, _ = standardize(X_all_e5)  # Standardize once before splitting.
X_train_e5, X_val_e5 = X_all_e5[:45], X_all_e5[45:]  # Split rows into train and validation.
y_train_e5, y_val_e5 = y_all_e5[:45], y_all_e5[45:]  # Split targets the same way.
print("train/val sizes:", len(y_train_e5), len(y_val_e5))  # Inspect split sizes.

▶ What you'll see: validation uses held-out rows, not training residuals.

In [ ]:
alphas_e5 = np.array([0.0, 0.25, 0.5, 0.75, 1.0])  # Compare ridge through lasso.
val_mse_e5 = []  # Store validation mean squared errors.
for alpha_e5 in alphas_e5:
    beta_e5, _ = fit_elastic_net_cd(X_train_e5, y_train_e5, lam=0.7, alpha=alpha_e5, steps=600)  # Fit one alpha.
    pred_val_e5 = X_val_e5 @ beta_e5  # Predict held-out rows.
    val_mse_e5.append(float(np.mean((y_val_e5 - pred_val_e5) ** 2)))  # Store validation MSE.
print("validation MSE:", np.round(val_mse_e5, 3))  # Inspect model-selection evidence.

In [ ]:
best_alpha_e5 = float(alphas_e5[int(np.argmin(val_mse_e5))])  # Choose the lowest validation error.
print("best alpha:", best_alpha_e5)  # Inspect the selected blend.
assert best_alpha_e5 in alphas_e5

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(alphas_e5, val_mse_e5, marker="o", color="navy")
plt.axvline(best_alpha_e5, color="red", linestyle="--", label=f"best α={best_alpha_e5}")
plt.title("Easy 5: choose alpha by validation")
plt.xlabel("alpha")
plt.ylabel("validation MSE")
plt.legend()
plt.show()

▶ What you'll see: one blend gives the lowest held-out error on this small split.

👀 Takeaway: alpha should be tuned by future-facing validation evidence, not by intuition alone.

## 🔴 Advanced

### Advanced 1 — Build a two-dimensional hyperparameter grid

**Goal.** Search lambda and alpha together, because Elastic Net has one knob for strength and another for penalty shape. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(11)  # Create reproducible data.
X_a1_raw = rng_a1.normal(size=(80, 6))  # Six candidate features.
X_a1_raw[:, 1] = X_a1_raw[:, 0] + 0.08 * rng_a1.normal(size=80)  # Correlated companion.
y_a1 = 1.3 * X_a1_raw[:, 0] + 1.1 * X_a1_raw[:, 1] - 0.8 * X_a1_raw[:, 4] + 0.4 * rng_a1.normal(size=80)  # Synthetic target.
X_a1, _, _ = standardize(X_a1_raw)  # Standardize for fair penalties.
Xtr_a1, Xva_a1 = X_a1[:60], X_a1[60:]  # Training and validation features.
ytr_a1, yva_a1 = y_a1[:60], y_a1[60:]  # Training and validation targets.
print("train/validation:", Xtr_a1.shape, Xva_a1.shape)  # Inspect split shapes.

▶ What you'll see: a held-out validation block for grid search.

In [ ]:
lams_a1 = np.array([0.05, 0.2, 0.8, 1.5])  # Strength grid.
alphas_a1 = np.array([0.0, 0.4, 0.8, 1.0])  # Blend grid.
val_grid_a1 = np.zeros((len(alphas_a1), len(lams_a1)))  # Store validation MSE for each pair.
for i_a1, alpha_a1 in enumerate(alphas_a1):
    for j_a1, lam_a1 in enumerate(lams_a1):
        beta_a1, _ = fit_elastic_net_cd(Xtr_a1, ytr_a1, lam=lam_a1, alpha=alpha_a1, steps=700)
        val_grid_a1[i_a1, j_a1] = np.mean((yva_a1 - Xva_a1 @ beta_a1) ** 2)
print("validation grid:\n", np.round(val_grid_a1, 3))

In [ ]:
best_pos_a1 = np.unravel_index(int(np.argmin(val_grid_a1)), val_grid_a1.shape)  # Locate best grid cell.
best_alpha_a1 = float(alphas_a1[best_pos_a1[0]])  # Read best alpha.
best_lam_a1 = float(lams_a1[best_pos_a1[1]])  # Read best lambda.
print("best alpha/lambda:", best_alpha_a1, best_lam_a1)  # Inspect selected pair.
assert val_grid_a1[best_pos_a1] == np.min(val_grid_a1)

In [ ]:
plt.figure(figsize=(5, 3.6))
plt.imshow(val_grid_a1, cmap="viridis", aspect="auto")
plt.colorbar(label="validation MSE")
plt.xticks(range(len(lams_a1)), lams_a1)
plt.yticks(range(len(alphas_a1)), alphas_a1)
plt.scatter([best_pos_a1[1]], [best_pos_a1[0]], color="red", marker="x", s=100)
plt.title("Advanced 1: alpha-lambda grid")
plt.xlabel("lambda")
plt.ylabel("alpha")
plt.show()

▶ What you'll see: the red X marks the lowest validation error in the two-knob search.

👀 Takeaway: Elastic Net selection is a joint search over strength and blend.

### Advanced 2 — Visualize the penalty geometry

**Goal.** Compare L1, L2, and Elastic Net contours, because the shape of the penalty explains selection and grouping behavior. We build it in 3 steps.

In [ ]:
b0_a2 = np.linspace(-2, 2, 121)  # Grid for coefficient 0.
b1_a2 = np.linspace(-2, 2, 121)  # Grid for coefficient 1.
B0_a2, B1_a2 = np.meshgrid(b0_a2, b1_a2)  # Build a coefficient plane.
L1_a2 = np.abs(B0_a2) + np.abs(B1_a2)  # L1 diamond values.
L2_a2 = 0.5 * (B0_a2 ** 2 + B1_a2 ** 2)  # L2 circular values.
EN_a2 = 0.6 * L1_a2 + 0.4 * L2_a2  # Elastic Net blend.
print("grid shape:", EN_a2.shape)  # Inspect contour grid size.

▶ What you'll see: a dense coefficient plane for drawing penalty contours.

In [ ]:
center_val_a2 = float(EN_a2[60, 60])  # Penalty at beta=(0,0).
corner_val_a2 = float(EN_a2[-1, -1])  # Penalty near beta=(2,2).
print("center/corner EN penalty:", round(center_val_a2, 3), round(corner_val_a2, 3))  # Inspect scale.
assert round(center_val_a2, 3) == 0.000

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for axis_a2, Z_a2, title_a2 in zip(ax, [L1_a2, L2_a2, EN_a2], ["L1 diamond", "L2 circle", "Elastic Net blend"]):
    axis_a2.contour(B0_a2, B1_a2, Z_a2, levels=[0.5, 1.0, 2.0, 3.0], colors="navy")
    axis_a2.axhline(0, color="gray", linewidth=0.5)
    axis_a2.axvline(0, color="gray", linewidth=0.5)
    axis_a2.set_title(title_a2)
    axis_a2.set_xlabel("β0")
ax[0].set_ylabel("β1")
plt.tight_layout()
plt.show()

▶ What you'll see: L1 has sharp axis-aligned corners, L2 is rounded, and Elastic Net blends both shapes.

👀 Takeaway: penalty geometry is the visual reason Elastic Net can select features while staying smoother than lasso.

### Advanced 3 — Track train versus validation as lambda changes

**Goal.** Sweep lambda and compare training with validation error, because stronger penalties can sacrifice fit to improve future behavior. We build it in 4 steps.

In [ ]:
rng_a3 = np.random.default_rng(13)  # Create deterministic data.
X_a3_raw = rng_a3.normal(size=(70, 8))  # Eight candidate features.
y_a3 = 1.5 * X_a3_raw[:, 0] - 1.2 * X_a3_raw[:, 2] + 0.6 * rng_a3.normal(size=70)  # Noisy linear target.
X_a3, _, _ = standardize(X_a3_raw)  # Standardize for penalized regression.
Xtr_a3, Xva_a3 = X_a3[:50], X_a3[50:]  # Split features.
ytr_a3, yva_a3 = y_a3[:50], y_a3[50:]  # Split targets.
lams_a3 = np.array([0.01, 0.05, 0.2, 0.8, 2.0, 5.0])  # Penalty strengths.
print("lambda grid:", lams_a3)  # Inspect the sweep.

▶ What you'll see: the grid ranges from weak to strong regularization.

In [ ]:
train_mse_a3 = []  # Store training errors.
val_mse_a3 = []  # Store validation errors.
coef_l1_a3 = []  # Store coefficient L1 sizes.
for lam_a3 in lams_a3:
    beta_a3, _ = fit_elastic_net_cd(Xtr_a3, ytr_a3, lam=lam_a3, alpha=0.6, steps=700)
    train_mse_a3.append(float(np.mean((ytr_a3 - Xtr_a3 @ beta_a3) ** 2)))
    val_mse_a3.append(float(np.mean((yva_a3 - Xva_a3 @ beta_a3) ** 2)))
    coef_l1_a3.append(float(np.sum(np.abs(beta_a3))))
print("train MSE:", np.round(train_mse_a3, 3))
print("validation MSE:", np.round(val_mse_a3, 3))

In [ ]:
best_lam_a3 = float(lams_a3[int(np.argmin(val_mse_a3))])  # Select lambda by validation error.
print("best validation lambda:", best_lam_a3)  # Inspect selected penalty strength.
assert best_lam_a3 in lams_a3

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(lams_a3, train_mse_a3, marker="o", label="train")
plt.plot(lams_a3, val_mse_a3, marker="o", label="validation")
plt.xscale("log")
plt.axvline(best_lam_a3, color="red", linestyle="--", label="best λ")
plt.title("Advanced 3: regularization tradeoff")
plt.xlabel("lambda")
plt.ylabel("MSE")
plt.legend()
plt.show()

▶ What you'll see: training error tends to rise with regularization, while validation can have a best middle value.

👀 Takeaway: the right penalty strength is chosen by generalization evidence, not by the lowest training error.

### Advanced 4 — Convert the lesson arithmetic into a decision audit

**Goal.** Recreate raw risk, cost, gap, relative gap, stabilized score, and final minimum in one auditable table, because the notebook should match the prose. We build it in 3 steps.

In [ ]:
losses_a4 = np.array([0.191, 0.148, 0.539])  # Verified losses.
cost_a4 = 0.090  # Verified method cost.
flexible_a4 = 0.419  # Verified flexible alternative score.
risk_a4 = float(losses_a4.mean())  # Raw empirical risk.
score_a4 = round(risk_a4, 3) + cost_a4  # Baseline full score using the rounded lesson risk.
print("risk:", round(risk_a4, 3), "score:", round(score_a4, 3))  # Inspect first decision pieces.

▶ What you'll see: the raw average and the cost-adjusted score are different.

In [ ]:
gap_a4 = flexible_a4 - score_a4  # Absolute evidence gap.
relative_gap_a4 = gap_a4 / flexible_a4  # Scale-aware evidence gap.
stable_a4 = 0.80 * score_a4  # Stabilizing knob reduces the score by 20%.
print("gap:", round(gap_a4, 3), "relative:", round(relative_gap_a4, 3), "stable:", round(stable_a4, 3))  # Inspect all lesson values.
assert round(risk_a4, 3) == 0.293
assert round(score_a4, 3) == 0.383
assert round(gap_a4, 3) == 0.036
assert round(relative_gap_a4, 3) == 0.086
assert round(stable_a4, 3) == 0.306

In [ ]:
labels_a4 = np.array(["baseline", "flexible", "stabilized"])
values_a4 = np.array([score_a4, flexible_a4, stable_a4])
winner_a4 = labels_a4[int(np.argmin(values_a4))]
print("winner:", winner_a4, "minimum:", round(float(np.min(values_a4)), 3))  # Inspect final decision.
assert winner_a4 == "stabilized"
plt.figure(figsize=(5, 3))
plt.bar(labels_a4, values_a4, color=["steelblue", "darkorange", "seagreen"])
plt.title("Advanced 4: decision audit")
plt.ylabel("full score")
plt.show()

▶ What you'll see: the stabilized option is the lowest valid full score.

👀 Takeaway: the lesson's correct unit of judgment is the full audited score.

### Advanced 5 — Bootstrap coefficient stability

**Goal.** Refit Elastic Net on bootstrap resamples and inspect coefficient variability, because stable models should not swing wildly when the sample changes a little. We build it in 4 steps.

In [ ]:
rng_a5 = np.random.default_rng(15)  # Create reproducible data and resamples.
n_a5 = 70  # Number of examples.
z_a5 = rng_a5.normal(size=n_a5)  # Shared correlated signal.
X_a5_raw = np.column_stack([z_a5 + 0.05 * rng_a5.normal(size=n_a5), z_a5 + 0.05 * rng_a5.normal(size=n_a5), rng_a5.normal(size=n_a5), rng_a5.normal(size=n_a5)])  # Correlated pair plus noise features.
y_a5 = 1.0 * X_a5_raw[:, 0] + 1.0 * X_a5_raw[:, 1] + 0.2 * rng_a5.normal(size=n_a5)  # Target uses the correlated pair.
X_a5, _, _ = standardize(X_a5_raw)  # Standardize for fitting.
print("data shape:", X_a5.shape)  # Inspect bootstrap dataset size.

▶ What you'll see: a four-feature regression problem with two correlated useful predictors.

In [ ]:
B_a5 = 20  # Number of bootstrap fits.
coefs_a5 = []  # Store coefficient vectors.
for b_a5 in range(B_a5):
    idx_a5 = rng_a5.integers(0, n_a5, size=n_a5)  # Sample rows with replacement.
    beta_a5, _ = fit_elastic_net_cd(X_a5[idx_a5], y_a5[idx_a5], lam=0.6, alpha=0.4, steps=500)  # Fit mixed Elastic Net.
    coefs_a5.append(beta_a5)  # Save coefficients.
coefs_a5 = np.array(coefs_a5)  # Convert to matrix: bootstrap by feature.
print("mean coefficients:", np.round(coefs_a5.mean(axis=0), 3))  # Inspect average fitted coefficients.

In [ ]:
std_a5 = coefs_a5.std(axis=0)  # Measure coefficient instability across resamples.
print("coefficient std:", np.round(std_a5, 3))  # Inspect variability.
assert np.all(std_a5 >= 0)

In [ ]:
plt.figure(figsize=(5, 3))
plt.errorbar(np.arange(coefs_a5.shape[1]), coefs_a5.mean(axis=0), yerr=std_a5, fmt="o", color="purple", capsize=4)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Advanced 5: bootstrap coefficient stability")
plt.xlabel("feature")
plt.ylabel("mean coefficient ± std")
plt.show()

▶ What you'll see: useful correlated features have nonzero average weight, and error bars show how much resampling moves them.

👀 Takeaway: Elastic Net is valued not only for fit, but for coefficient behavior that remains stable under sample noise.